In [ ]:
import numpy as np
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import viser

from lac.perception.segmentation import SemanticClasses
from lac.slam.backend import SemanticPointCloud
from lac.mapping.mapper import bin_points_to_grid, nanmedian_filter, process_map
from lac.mapping.map_utils import get_geometric_score, get_rocks_score
from lac.mapping.interpolation import interpolate_heights, interpolate_heights_rbf
from lac.utils.plotting import plot_heatmap, plot_surface, plot_rock_maps
from lac.util import load_data, get_positions_from_poses

%load_ext autoreload
%autoreload 2

# Load data


In [ ]:
# Load the data logs
# data_path = "/home/shared/data_raw/LAC/runs/triangles_preset6"
data_path = "/home/shared/data_raw/LAC/runs/2025-05-28_11-59-12"
# data_path = "../../output/NavAgent/triangles_preset6"
# data_path = "../../../output/NavAgent/2025-05-15_18-20-47"
initial_pose, lander_pose, poses, imu_data, cam_config, json_data = load_data(data_path)
print(f"Loaded {len(poses)} poses")

In [ ]:
ground_truth_map = np.load(Path(data_path) / "Moon_Map_01_6_rep0.dat", allow_pickle=True)
agent_map = np.load(Path(data_path) / "Moon_Map_01_6_rep0_agent.dat", allow_pickle=True)

In [ ]:
agent_map = ground_truth_map.copy()

In [ ]:
point_map = SemanticPointCloud.from_file(Path(data_path) / "semantic_points.npz")

# Height mapping


In [ ]:
ground_points = point_map.points[point_map.labels == SemanticClasses.GROUND.value]
ground_grid = bin_points_to_grid(ground_points, statistic="robust_mean")

In [ ]:
ground_grid = nanmedian_filter(ground_grid, size=3)

In [ ]:
agent_map[:, :, 2] = ground_grid

In [ ]:
plot_surface(agent_map)

In [ ]:
agent_map[:] = interpolate_heights(agent_map)

In [ ]:
plot_surface(agent_map)

In [ ]:
print(f"Geometric score: {get_geometric_score(ground_truth_map, agent_map)}")
print(f"Rocks score: {get_rocks_score(ground_truth_map, agent_map)}")

# Rock mapping


In [ ]:
plot_rock_maps(ground_truth_map, agent_map)

In [ ]:
agent_map = process_map(point_map, agent_map)

In [ ]:
print(f"Rocks score: {get_rocks_score(ground_truth_map, agent_map)}")

# Semantic Points


In [ ]:
semantic_points = SemanticPointCloud.from_file(Path(data_path) / "semantic_points.npz")
semantic_points.points.shape

In [ ]:
# Downsample points by taking every 10th point
downsampled_points = semantic_points.points[::10]
downsampled_labels = semantic_points.labels[::10]

# Filter out sky points (label 4)
valid_mask = (downsampled_labels != 4) & (downsampled_labels != 0)
downsampled_points = downsampled_points[valid_mask]
downsampled_labels = downsampled_labels[valid_mask]


# Create color mapping
colors = {
    1: [1.0, 0.0, 0.0],  # red for rocks
    2: [1.0, 0.843, 0.0],  # gold for lander
    3: [0.5, 0.5, 0.5],  # gray for ground
}

# Convert labels to colors
point_colors = np.array([colors[label] for label in downsampled_labels])

# Plot in viser
if "server" not in globals():
    server = viser.ViserServer(port=8080)

background_img = np.zeros((1080, 1920, 3), dtype=np.uint8)  # RGB black
server.scene.set_background_image(background_img, format="jpeg")
server.scene.add_point_cloud(
    "/semantic_cloud",
    points=downsampled_points.astype(np.float32),
    colors=point_colors,
    point_size=0.01,
)

# # Add the trajectory
# traj = get_positions_from_poses(poses)
# segments = np.stack([traj[:-1], traj[1:]], axis=1)
# server.scene.add_line_segments(
#     "/trajectory",
#     points=segments,
#     colors=np.array([173, 216, 230], dtype=np.uint8),
# )

In [ ]:
data = server.get_scene_serializer().serialize()  # Returns bytes
Path("recording.viser").write_bytes(data)